In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Importing Data

In [ ]:
conn =sqlite3.connect('customer_churn.db')
sql_query=''' 
         SELECT name
         FROM sqlite_master
         WHERE type='table';
'''
tables=pd.read_sql(sql_query,conn)
# creating dataframe for each table
for table_name in tables['name']:
    df=pd.read_sql(f"SELECT * from {table_name}",conn)
    globals()[f'df_{table_name}']=df
    print(f'Created dataframe : df_{table_name}')

    df_db_customer: pd.DataFrame
    df_db_subscription: pd.DataFrame
    df_db_support: pd.DataFrame

conn.close()

# Data Cleaning

In [ ]:
# Customer DataFrame

In [ ]:
df_db_customer.head()
df_db_customer.tail()

In [ ]:
# rename column name - name
df_db_customer.rename(columns={'name':'customer_name'})

In [ ]:
# dropping column - interests and pincode
df_db_customer.drop(columns=['interests','pincode'],inplace=True)
df_db_customer

In [ ]:
# change datatype - dob
(df_db_customer['dob'])=pd.to_datetime(df_db_customer['dob'])

In [ ]:
# Data standarization
df_db_customer['gender'].unique()
df_db_customer['gender']=df_db_customer['gender'].replace({'Men' : 'Male','Women' : 'Female'})

In [ ]:
# fixing missing values - country
df_db_customer[df_db_customer['country'].isna()]
# country and state - unique value pair

# Creating state → country map from non-null rows
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

# Fill the missing country using State
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [ ]:
df_db_customer[df_db_customer['country'].isna()] # no null value in country col
df_db_customer

In [ ]:
# Subscription DataFrame

In [ ]:
df_db_subscription.info()

In [ ]:
# changing datatype from string to datetime.
date_col=['subscription_start_date','renewal_date','cancellation_date']
df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)
df_db_subscription.info()


In [ ]:
# Support DataFrame

In [ ]:
df_db_support.head()

In [ ]:
# Dropping unwanted columns --> col_1 and comment.
df_db_support.drop(columns=['col_1','comment'],inplace=True)

In [ ]:
# Changing complaint_date datatype to datetime.
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])
df_db_support.info()

# Feature Engineering and Data Analysis

In [ ]:
# create a new col using exesting col - churn flag
df_db_subscription['churn flag']=np.where(df_db_subscription['cancellation_date'].notna(),1,0)
df_db_subscription.head()

In [ ]:
# first fix support table duplicates then merge
df=(df_db_subscription
                .merge(df_db_customer , on= 'customerid',how='left')
                .merge(df_db_support , on= 'customerid' , how='left'))


In [ ]:
df_db_support['complaint_count']=df_db_support.groupby('customerid')['customerid'].transform('count')
df_db_support=df_db_support.sort_values('complaint_date').drop_duplicates('customerid',keep='last')
df_db_support

In [ ]:
# merge df
df=(df_db_subscription
                .merge(df_db_customer , on= 'customerid',how='left')
                .merge(df_db_support , on= 'customerid' , how='left'))

In [ ]:
# exporting final data and making it csv file
df.to_csv('Exported_churn_data.csv',index=False)

### Data Analysis

In [ ]:
# 1. churn rates
churn_rate=df['churn flag'].mean()*100
print('churn_rate =',round(churn_rate,2),'%')

In [ ]:
# 2. Retention rate
Retention_rate = 100-churn_rate
print('Retention_rate =',round(Retention_rate,2),'%')


In [ ]:
# 3. Churn by plan_type
churn_by_plan=df.groupby("plan_type")['churn flag'].mean().mul(100).round(2).reset_index(name='churn_rate %')
print(churn_by_plan)

In [ ]:
# 4. ARPU - Average revenue per users
arpu = df['monthly_charges'].mean()
print('Average revenue per user =',round(arpu,2))

In [ ]:
# 5. calculate avg customer age
today_date=pd.Timestamp.today()
df['dob'] = pd.to_datetime(df['dob'])
age=today_date.year - df['dob'].dt.year
avg_age=age.mean()
print('Average customer age =',round(avg_age,2),'Years')


In [ ]:
# 6. Avg Customer Tenure
today = pd.Timestamp.today()

df['tenure_days'] = np.where(
        df['cancellation_date'].notna(),

    (df['cancellation_date'] - df['subscription_start_date']).dt.days,

    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure_days'].mean()
print("Avg Tenure (Days) = ", round(avg_tenure))

In [ ]:
# 7. Revenue at risk - revenue lost from churned users
revenue_at_risk=df.loc[df['churn flag'] == 1,'monthly_charges'].sum()
print('Revenue at risk ( Rs k ) =',revenue_at_risk)


In [ ]:
# 8. Esclation Rate
esclation_rate=(df['escalations']=='Y').mean()*100
print('Esclation Rate = ',round(esclation_rate,2),'%')

In [ ]:
# 9. Average complaint per user
avg_complaints=df['complaint_count'].sum() /df['customerid'].nunique()
print('Average complaints per user =',round(avg_complaints,2))

In [ ]:
# 10. Correlation Esclation vs churn
# df['escalations'] = np.where(df['escalations']== 'Y' ,1,0)  --> i.e encoding string to int type

df['escalations'] = np.where(df['escalations']== 'Y' ,1,0)

corr_df = df[['escalations','churn flag']].dropna()

correlation = corr_df['escalations'].corr(df['churn flag'])
print('Correlation between escalation vs churn =',round(correlation,2))

In [ ]:
# 11. Churn risk -- create a column using existing column
conditions=[
    (df['churn_score'] < 50),
    (df['churn_score'] >= 50) & (df['churn_score'] < 70),
    (df['churn_score'] >= 70)
]
choices = ['low','mid','high']
df['churn_risk']=np.select(conditions,choices,default='unknown')
df

# Data Visualization

### Using Matplotlib 

In [ ]:
df_visual=df.copy()

In [ ]:
# 1. Monthly churn trend  -- (Time Series Kpi)
df_visual['cancellation_month']=df_visual['cancellation_date'].dt.to_period('M')

churn_trend = df_visual[df_visual['churn flag'] ==1].groupby('cancellation_month').size()

plt.figure(figsize=(6,3))

plt.plot(churn_trend.index.astype('str'), churn_trend.values , color ='blue' , marker='o' , linestyle ='dashed', linewidth= 2 , markersize= 6 )
plt.title("Monthly Churn Trend")
plt.xlabel('Months')
plt.ylabel('Churned Customers')
plt.show()

In [ ]:
churn_plan = df_visual.groupby('plan_type')['churn flag'].mean()
churn_plan


In [ ]:
# 2. Churn by plan type
churn_plan = df_visual.groupby('plan_type')['churn flag'].mean()

colors=plt.cm.Set2(np.linspace(0,1,len(churn_plan)))
plt.figure(figsize=(7,4))

plt.bar(churn_plan.index, churn_plan.values, color = colors)
plt.show()

In [ ]:
# 3. Churn by States
churn_plan = df_visual.groupby('state')['churn flag'].mean()

colors=plt.cm.Set2(np.linspace(0,1,len(churn_plan)))
plt.figure(figsize=(11,4))

plt.bar(churn_plan.index, churn_plan.values, color = colors)
plt.show()

### Using Seaborn

In [ ]:
# encoding --> Convert str to numeric values 


df_encoded = df_visual[['plan_type', 'contract_type', 'churn_score', 'churn flag', 'churn_risk', 'escalations']]

order_mappings={
    'plan_type' : ['Basic', 'Standard', 'Premium'],
    'contract_type' : ['Monthly', 'Annual'],
    'churn_risk' : ['low', 'high', 'mid']
}
for col,order in order_mappings.items():
    df_encoded[col]=pd.Categorical(df_encoded[col].astype('category'), categories=order , ordered=True).codes

In [ ]:
# 1. Heatmap
sns.heatmap(df_encoded.corr(), annot=True)

In [ ]:
# 2. Pairplot  --> to find relationship in a dataset
sns.pairplot(df_encoded)

In [ ]:
# 3. catplot / facegrid plot

sns.catplot(data=df_visual,
            x='plan_type',
            y='monthly_charges',
            hue='gender',
            col='churn_risk')


## Pivot Table

In [ ]:
df_visual.columns

In [ ]:
# Pivot Table

pd.pivot_table(
    df_visual,
    index='plan_type',
    values='churn flag',
    aggfunc = 'mean'
)    

In [ ]:
# pivot table using multiple cols and agg type

pd.pivot_table(
    df_visual,
    index='plan_type',
    values=['monthly_charges', 'customerid', 'churn flag'],
    aggfunc = {
        'monthly_charges' : 'sum',
        'customerid' : 'nunique',
        'churn flag' : 'mean'
    }
)